# Example 3: Red Zone Playcalling Analysis

Tasks

* Compare EPA/play inside 20 yards vs outside 20 yards
* Look at run/pass ratio shift
* Touchdown rate vs League average

Hypothesis 1:
* Do teams perform worse (EPA/play) inside the red zone (<= 20 yards)?

Hypothesis 2:
* Do teams change their behavior (run vs pass) in the red zone?

In [15]:
# Imports
import nfl_data_py as nfl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np
from statsmodels.sandbox.regression.sympy_diff import df

from dashboards.epa_dashboard.epa_dashboard_dataframes import team_avg_epa

In [16]:
# Get play-by-play dataframe
pbp = nfl.import_pbp_data([2025])
pbp.head()

2025 done.
Downcasting floats.


,play_id,game_id,old_game_id_x,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
0,1.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,,,None,Denzel Burke;Joey Blount;Dadrion Taylor-Demers...,Rejzohn Wright;Quincy Riley;Ugo Amadi;Julian B...,CB;FS;FS;ILB;ILB;RB;S;TE;TE;TE;WR,CB;CB;FS;FS;ILB;ILB;K;MLB;OLB;RB;SS,29;32;42;44;50;31;36;84;81;87;4,25;29;0;23;53;44;19;28;40;13;33
2,63.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,,,None,Hjalte Froholdt;Evan Brown;Isaiah Adams;Kyler ...,Isaac Yiadom;Kool-Aid McKinstry;Nathan Shepher...,C;G;G;QB;RB;T;T;TE;TE;TE;WR,CB;CB;DT;DT;FS;ILB;MLB;NT;OLB;OLB;SS,72;63;74;1;6;73;70;85;84;87;18,27;4;93;90;23;20;56;92;94;96;21
3,85.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,QUICK OUT,ZONE_COVERAGE,COVER_2,Hjalte Froholdt;Evan Brown;Isaiah Adams;Kyler ...,Isaac Yiadom;Kool-Aid McKinstry;Nathan Shepher...,C;G;G;QB;RB;T;T;TE;TE;WR;WR,CB;CB;DT;DT;FS;ILB;MLB;NT;OLB;OLB;SS,72;63;74;1;6;73;70;85;87;14;18,27;4;93;90;23;20;56;92;94;96;21
4,115.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,1.0,,ZONE_COVERAGE,COVER_6,Hjalte Froholdt;Evan Brown;Isaiah Adams;Kyler ...,Isaac Yiadom;Kool-Aid McKinstry;Nathan Shepher...,C;G;G;QB;RB;T;T;TE;TE;WR;WR,CB;CB;DT;DT;FS;ILB;MLB;NT;OLB;OLB;SS,72;63;74;1;6;73;70;85;87;17;18,27;4;93;90;23;20;56;92;94;96;21


In [17]:
# Keep only the columns we need and make a copy of original df
pbp_small = pbp[[
    'down',
    'ydstogo',
    'epa',
    'play_type',
    'posteam',
    'yardline_100',
    'game_id',
    'play_id'
]].copy()

pbp_small.head()

,down,ydstogo,epa,play_type,posteam,yardline_100,game_id,play_id
0,NaN,0.0,-0.000000,None,None,NaN,2025_01_ARI_NO,1.0
1,NaN,0.0,-0.352700,kickoff,ARI,35.0,2025_01_ARI_NO,40.0
2,1.0,10.0,-0.190052,run,ARI,78.0,2025_01_ARI_NO,63.0
3,2.0,7.0,1.317340,pass,ARI,75.0,2025_01_ARI_NO,85.0
4,1.0,10.0,-1.694360,pass,ARI,64.0,2025_01_ARI_NO,115.0


In [18]:
# Filter to run and pass plays and clean missing values
pbp_small = pbp_small[
    (pbp_small['play_type'].isin(['run', 'pass'])) &
    (pbp_small['epa'].notna()) &
    (pbp_small['posteam'].notna())
]

pbp_small.shape

(34628, 8)

In [19]:
# Define the red zone
# This gives a boolean value, True if in the redzone
pbp_small['red_zone'] = pbp_small['yardline_100'] <= 20
pbp_small.head()

,down,ydstogo,epa,play_type,posteam,yardline_100,game_id,play_id,red_zone
2,1.0,10.0,-0.190052,run,ARI,78.0,2025_01_ARI_NO,63.0,False
3,2.0,7.0,1.317340,pass,ARI,75.0,2025_01_ARI_NO,85.0,False
4,1.0,10.0,-1.694360,pass,ARI,64.0,2025_01_ARI_NO,115.0,False
5,2.0,21.0,-1.284150,run,ARI,75.0,2025_01_ARI_NO,135.0,False
6,3.0,23.0,-0.840574,run,ARI,77.0,2025_01_ARI_NO,166.0,False


## Compare EPA per Play

In [20]:
# Get average epa by red zone label
epa_comparison = pbp_small.groupby('red_zone')['epa'].mean()
epa_comparison.head()

red_zone
False    0.005320
True     0.001609
Name: epa, dtype: float32

In [29]:
epa_comparison_viz = epa_comparison.reset_index()
epa_comparison_viz['red_zone'] = epa_comparison_viz['red_zone'].map(
    {
        True: 'Red Zone',
        False: 'Outside Red Zone'
    }
)

px.bar(
    epa_comparison_viz,
    x='red_zone',
    y='epa',
    title="EPA per Play: Red Zone vs Outside Red Zone"
)

EPa is lower in the red zone suggesting that offenses are less productive in the end zone. This could be due to less field space (so less options for routes), a more compressed defense, and less explosive plays.

## Run vs. Pass Ratio

In [21]:
playcalling = (
    pbp_small.groupby(['red_zone', 'play_type'])
    .size() # Counts rows in each group
    .groupby(level=0)
    .apply(lambda x: x / x.sum())
    .unstack() # pivot one index level into columns
)

playcalling

C:\Users\viole\AppData\Local\Temp\ipykernel_612\1395135645.py:5: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda x: x / x.sum())


play_type,pass,run
red_zone,,
False,0.580903,0.419097
True,0.511881,0.488119


Teams typically pass more in and out of the red zone. However, in the red zone that differential decreases.

## EPA by Play Type

In [22]:
epa_by_play_type = pbp_small.groupby(['red_zone', 'play_type'])['epa'].mean().unstack()

epa_by_play_type

play_type,pass,run
red_zone,,
False,0.020490,-0.015707
True,-0.026541,0.031130


EPA for rushing significantly increases in the red zone, but epa for passing decreases.

## Team Analysis

In [28]:
team_epa = (
    pbp_small.groupby(['posteam', 'red_zone'])['epa']
    .mean()
    .unstack()
    .reset_index()
)

team_epa['differential'] = team_epa[False] - team_epa[True]

team_epa

red_zone,posteam,False,True,differential
0,ARI,-0.004956,-0.091053,0.086097
1,ATL,-0.029443,0.029461,-0.058904
2,BAL,0.075898,-0.150634,0.226533
3,BUF,0.136336,0.142879,-0.006543
4,CAR,-0.042260,-0.036098,-0.006161
5,CHI,0.088774,-0.002657,0.091431
6,CIN,-0.021034,0.066361,-0.087395
7,CLE,-0.205463,-0.084054,-0.121410
8,DAL,0.133299,-0.069092,0.202392
9,DEN,0.016629,0.112410,-0.095782
